<a href="https://colab.research.google.com/github/yoshiagent/automaticControl-colab/blob/main/notebooks/LeadLag_Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 頻域補償器設計 Lead／Lag Compensator Design

對照網頁：[自動控制｜相位領先補償器](https://yoshiagent.github.io/automaticControl/FRD-Lead.html)、
[相位落後補償器](https://yoshiagent.github.io/automaticControl/FRD-Lag.html)、
[增益邊限與相位邊限](https://yoshiagent.github.io/automaticControl/FRD-GainPhaseMargin.html)

原課程在頻域設計時用兩支小程式查表：`Leadtab.m` 列出 lead 補償器 $\alpha$ 與最大相位領先角 $\phi_{\max}$ 的對照表，
`mo2zeta2pm.m` 把超越量規格換成阻尼比與相位邊限。這本筆記本先列出原本的 MATLAB 程式，再給對應的 Python 版本，
最後用頻率響應實際算出相位邊限，驗算網頁上的 lead 與 lag 設計例（原課程用 MATLAB SISO Design Tool 讀出的數值）。

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `Leadtab.m` | $\alpha$、$10\log\alpha$、$\phi_{\max}=\sin^{-1}\frac{\alpha-1}{\alpha+1}$ 對照表 | 第 1 節 |
| `mo2zeta2pm.m` | 超越量 $M_p$ → 阻尼比 $\zeta$ → 相位邊限 PM | 第 2 節 |
| （網頁設計例） | $G(s)=\dfrac{50}{s(1+0.2s)}$ 的 lead 與 lag 補償，要求 PM > 48° | 第 3、4 節 |

只用 Colab 內建的 numpy、scipy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
from scipy import signal
from scipy.optimize import brentq
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show(fig, title, xlab=None, ylab=None, h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def G_jw(num, den, w):
    """G(jw)：num、den 為 s 的多項式係數（高次在前，同 MATLAB）"""
    s = 1j * np.asarray(w, dtype=float)
    return np.polyval(num, s) / np.polyval(den, s)


def margin(num, den, wlo=1e-3, whi=1e3):
    """仿 MATLAB margin 的相位邊限部分：增益交越頻率 wg（|G|=1）與 PM = 180° + ∠G(jwg)"""
    w = np.logspace(np.log10(wlo), np.log10(whi), 20001)
    lm = np.log(np.abs(G_jw(num, den, w)))
    i = np.where(np.diff(np.sign(lm)))[0][0]
    wg = brentq(lambda x: np.log(abs(G_jw(num, den, x))), w[i], w[i + 1])
    ph = np.degrees(np.angle(G_jw(num, den, wg)))
    return wg, 180 + ph

## 1. Lead 補償器的 $\alpha$ 對照表 `Leadtab.m`

```matlab
da=0.25;
a=1:da:10;
halfadb=10*log10(a);
phimax=asin((a-1)./(a+1))*180/pi;
tab=[a' halfadb' phimax']
```

Lead 補償器 $G_c(s)=\dfrac{1+\alpha\tau s}{1+\tau s}$（$\alpha>1$）在 $\omega_m=\dfrac{1}{\tau\sqrt\alpha}$ 有最大相位領先角
$\sin\phi_{\max}=\dfrac{\alpha-1}{\alpha+1}$，此處增益為 $20\log\sqrt\alpha=10\log\alpha$ dB（推導見網頁「Lead_1」）。

In [2]:
da = 0.25
a = np.arange(1, 10 + da / 2, da)
halfadb = 10 * np.log10(a)
phimax = np.degrees(np.arcsin((a - 1) / (a + 1)))
tab = np.column_stack([a, halfadb, phimax])
print(f"tab: {tab.shape[0]} rows (a = 1:0.25:10)")
print("   alpha   10log(alpha)   phimax(deg)")
for r in tab:
    if r[0] == int(r[0]):                                           # 只印整數 alpha（網頁上的表）
        print(f"{r[0]:8.2f} {r[1]:12.2f} {r[2]:13.2f}")

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_scatter(x=a, y=phimax, name="phi_max (deg)", line=dict(color="#d62728", width=2.4))
fig.add_scatter(x=a, y=halfadb, name="10 log(alpha) (dB)", line=dict(color="#1f77b4", width=2.4), secondary_y=True)
fig.update_yaxes(title_text="phi_max (deg)", secondary_y=False)
fig.update_yaxes(title_text="gain at w_m (dB)", secondary_y=True)
show(fig, "Leadtab.m: maximum phase lead vs alpha", xlab="alpha")

tab: 37 rows (a = 1:0.25:10)
   alpha   10log(alpha)   phimax(deg)
    1.00         0.00          0.00
    2.00         3.01         19.47
    3.00         4.77         30.00
    4.00         6.02         36.87
    5.00         6.99         41.81
    6.00         7.78         45.58
    7.00         8.45         48.59
    8.00         9.03         51.06
    9.00         9.54         53.13
   10.00        10.00         54.90


網頁上的表 $\alpha=4$ 寫 36.86、$\alpha=5$ 寫 6.98、$\alpha=8$ 寫 51.05；精確值是 $36.8699^\circ$、$6.9897$ dB、$51.0576^\circ$，
四捨五入應為 36.87、6.99、51.06（原表是直接截去第三位小數），不影響查表選 $\alpha$。

## 2. 超越量 → 阻尼比 → 相位邊限 `mo2zeta2pm.m`

```matlab
mp=0.2;
lnmp=log(mp);
zeta=-lnmp/sqrt(pi^2+lnmp^2)
pm=atan(2*zeta/sqrt(-2*zeta^2+sqrt(4*zeta^4+1)))*180/pi
```

標準二階系統 $T(s)=\dfrac{\omega_n^2}{s^2+2\zeta\omega_ns+\omega_n^2}$ 的超越量 $M_p=e^{-\zeta\pi/\sqrt{1-\zeta^2}}$，反解得
$\zeta=\dfrac{-\ln M_p}{\sqrt{\pi^2+\ln^2M_p}}$；它的開迴路 $G(s)=\dfrac{\omega_n^2}{s(s+2\zeta\omega_n)}$ 的相位邊限
$PM=\tan^{-1}\dfrac{2\zeta}{\sqrt{-2\zeta^2+\sqrt{4\zeta^4+1}}}$。下面另外用步階響應與頻率響應實際驗算。

In [3]:
mp = 0.2
lnmp = np.log(mp)
zeta = -lnmp / np.sqrt(np.pi**2 + lnmp**2)
pm = np.degrees(np.arctan(2 * zeta / np.sqrt(-2 * zeta**2 + np.sqrt(4 * zeta**4 + 1))))
print(f"zeta = {zeta:.4f}")
print(f"pm = {pm:.2f} deg")

wn = 1.0
tt = np.linspace(0, 15, 1501)
_, y = signal.step(([wn**2], [1, 2 * zeta * wn, wn**2]), T=tt)
print(f"check: step overshoot of T(s) = {100 * (y.max() - 1):.2f} %")
wg, pm_num = margin([wn**2], [1, 2 * zeta * wn, 0])
print(f"check: PM of G(s) = wn^2/(s(s+2 zeta wn)) = {pm_num:.2f} deg at wg = {wg:.4f} wn")

fig = go.Figure()
fig.add_scatter(x=tt, y=y, name="step response")
fig.add_hline(y=1.2, line=dict(dash="dot", color="#d62728"), annotation_text="Mp = 20 %")
show(fig, f"Standard 2nd-order system, zeta = {zeta:.3f}", xlab="t (s), wn = 1", ylab="y(t)")

zeta = 0.4559
pm = 48.15 deg
check: step overshoot of T(s) = 20.00 %
check: PM of G(s) = wn^2/(s(s+2 zeta wn)) = 48.15 deg at wg = 0.8168 wn


## 3. 驗算網頁的 lead 設計例

受控體 $G(s)=\dfrac{K}{s(1+0.2s)}$，要求 $e_{ss}<0.02$（斜坡輸入）⇒ $K_v=K>50$，取 $K=50$；要求 $PM>48^\circ$。
網頁的步驟：未補償 PM ≈ 18° ⇒ 需補 30° ⇒ 查第 1 節的表取 $\alpha=4$（36.87°）⇒ 在未補償系統 $-10\log\alpha=-6.02$ dB 處找新的
增益交越頻率 $\omega_m$ ⇒ $\tau=\dfrac{1}{\omega_m\sqrt\alpha}$。

In [4]:
num0, den0 = [50], [0.2, 1, 0]
wg0, pm0 = margin(num0, den0)
print(f"uncompensated: PM = {pm0:.1f} deg at wg = {wg0:.2f} rad/s")
alpha = 4
wm = brentq(lambda w: 20 * np.log10(abs(G_jw(num0, den0, w))) + 10 * np.log10(alpha), 1, 100)
tau = 1 / (wm * np.sqrt(alpha))
print(f"|G| = -10log(alpha) = -6.02 dB at wm = {wm:.1f} rad/s -> tau = {tau:.4f}")
num_lead = np.polymul(num0, [0.088, 1])                             # 網頁取 tau = 0.022：Gc = (1+0.088s)/(1+0.022s)
den_lead = np.polymul(den0, [0.022, 1])
wg1, pm1 = margin(num_lead, den_lead)
print(f"lead compensated: PM = {pm1:.1f} deg at wg = {wg1:.1f} rad/s")

uncompensated: PM = 18.0 deg at wg = 15.42 rad/s
|G| = -10log(alpha) = -6.02 dB at wm = 22.1 rad/s -> tau = 0.0226
lead compensated: PM = 49.8 deg at wg = 21.8 rad/s


## 4. 驗算網頁的 lag 設計例

同一個受控體與規格，改用 lag 補償器 $G_c(s)=\dfrac{1+\tau s}{1+\alpha\tau s}$（$\alpha>1$）：
PM 要求 48° 再加 5° 餘裕 ⇒ 找未補償系統相位 $=-180^\circ+53^\circ=-127^\circ$ 的頻率當新的增益交越頻率 $\omega_g'$ ⇒
該處增益 $20\log\alpha$ 要被補償器壓下來 ⇒ 零點放在 $\omega_g'/10$：$\tau=10/\omega_g'$。

In [5]:
wgp = brentq(lambda w: np.degrees(np.angle(G_jw(num0, den0, w))) + 127, 0.1, 100)
alpha_lag = abs(G_jw(num0, den0, wgp))
tau_lag = 10 / wgp
print(f"phase = -127 deg at wg' = {wgp:.2f} rad/s; 20log(alpha) = {20 * np.log10(alpha_lag):.1f} dB -> alpha = {alpha_lag:.1f}; tau = {tau_lag:.2f}")
num_lag = np.polymul(num0, [2.67, 1])                               # 網頁：Gc = (1+2.67s)/(1+28.57s)
den_lag = np.polymul(den0, [28.57, 1])
wg2, pm2 = margin(num_lag, den_lag)
print(f"lag compensated: PM = {pm2:.1f} deg at wg = {wg2:.2f} rad/s")

phase = -127 deg at wg' = 3.77 rad/s; 20log(alpha) = 20.5 dB -> alpha = 10.6; tau = 2.65
lag compensated: PM = 47.9 deg at wg = 3.75 rad/s


網頁上的 $\omega_m=22.2$、$\omega_g'=3.74$、$\alpha=10.7$、$\tau=2.67$ 是從 MATLAB Bode 圖游標讀出的，和上面精確計算的 22.1、3.77、10.6、2.65 差在讀圖誤差；
網頁取的數值（$\tau=0.022$、$G_c=\frac{1+2.67s}{1+28.57s}$）算出的 PM 與 SISO Design Tool 的 49.8°、47.9° 完全一致。

兩種補償都達到 PM ≈ 48°～50°，但做法相反：lead 把交越頻率推高（22 rad/s，反應快），lag 把交越頻率拉低（3.7 rad/s，反應慢但高頻雜訊衰減多）。
下面把三者的 Bode 圖與閉迴路步階響應畫在一起（和網頁上 SISO Design Tool 的結果對照）。

In [6]:
w = np.logspace(-3, 3, 600)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06)
cases = [("uncompensated", num0, den0, "black"), ("lead", num_lead, den_lead, "#d62728"), ("lag", num_lag, den_lag, "#1f77b4")]
for name, n, d, c in cases:
    G = G_jw(n, d, w)
    fig.add_scatter(x=w, y=20 * np.log10(np.abs(G)), name=name, line=dict(color=c), row=1, col=1)
    fig.add_scatter(x=w, y=np.degrees(np.unwrap(np.angle(G))), name=name, line=dict(color=c), showlegend=False, row=2, col=1)
fig.add_hline(y=-180, line=dict(dash="dot", color="gray"), row=2, col=1)
fig.update_xaxes(type="log", title_text="w (rad/s)", row=2, col=1)
fig.update_yaxes(title_text="magnitude (dB)", row=1, col=1)
fig.update_yaxes(title_text="phase (deg)", range=[-200, -80], row=2, col=1)
show(fig, "Open-loop Bode: 50/(s(1+0.2s)) with lead / lag compensation", h=620)

tt = np.linspace(0, 6, 601)
fig = go.Figure()
for name, n, d, c in cases:
    cl_den = np.polyadd(d, n)                                       # 單位負回授：T = n / (d + n)
    _, y = signal.step((n, cl_den), T=tt)
    print(f"{name:14s}: closed-loop overshoot = {100 * (y.max() - 1):5.1f} %")
    fig.add_scatter(x=tt, y=y, name=name, line=dict(color=c))
show(fig, "Closed-loop step response", xlab="t (s)", ylab="y(t)")

uncompensated : closed-loop overshoot =  60.5 %
lead          : closed-loop overshoot =  22.5 %
lag           : closed-loop overshoot =  24.1 %
